In [1]:
import pandas as pd

## Hybridization Control Normalization

In [2]:
TYPE_HCE = "Hybridization Control Elution"

In [3]:
features = pd.read_csv(
    "/mnt/data/processed/features.csv"
)
samples = pd.read_csv(
    "/mnt/data/processed/samples.csv"
)
measurements_raw = pd.read_csv(
    "/mnt/data/processed/measurements.raw.csv"
)

In [4]:
features_hce = features.loc[
    features["Type"] == TYPE_HCE
]

In [5]:
measurements_hce = (
    features_hce["ProbeId"]
                .to_frame()
                .join(
                    measurements_raw.set_index("ProbeId"),
                    on = "ProbeId"
                )
)

In [6]:
measurements_hce_ref = (
    measurements_hce.groupby(["PlateId", "ProbeId"])
                     .value
                     .median()
                     .rename("value_ref")
)

In [7]:
measurements_hce_scale_factor = measurements_hce.join(
    measurements_hce_ref,
    on = ["PlateId", "ProbeId"]
)
measurements_hce_scale_factor["value_scale_factor"] = (
    measurements_hce_scale_factor.value_ref /
       measurements_hce_scale_factor.value
)
measurements_hce_scale_factor = (
    measurements_hce_scale_factor.groupby(["PlateId", "PlatePosition"])
                             .value_scale_factor
                             .median()
)

In [8]:
measurements_hcn = measurements_raw.join(
    measurements_hce_scale_factor,
    on = ["PlateId", "PlatePosition"]
)
measurements_hcn.value *= measurements_hcn.value_scale_factor
measurements_hcn = measurements_hcn.drop("value_scale_factor", axis = 1)
measurements_hcn.to_csv(
    "/mnt/data/processed/measurements.hybridization_control_normalized.csv",
    index=False
)

measurements_hcn

,PlateId,PlatePosition,ProbeId,value
0,P0031168,A1,10000-28,628.862531
1,P0031168,A10,10000-28,621.353074
2,P0031168,A11,10000-28,550.560857
3,P0031168,A12,10000-28,557.348126
4,P0031168,A2,10000-28,394.584597
...,...,...,...,...
15571795,P0031201,H5,9999-1,2903.102446
15571796,P0031201,H6,9999-1,1622.180841
15571797,P0031201,H7,9999-1,7548.129488
15571798,P0031201,H8,9999-1,6344.403187
